In [7]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

!pip install scipy
from scipy import stats
from pathlib import Path


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
BASE_DIR = Path().resolve()

if (BASE_DIR / "data").exists():
    DATA_DIR = BASE_DIR / "data"
else:
    DATA_DIR = BASE_DIR.parent / "data"


csv_path = DATA_DIR / "topic_frequency.csv"

df = pd.read_csv(csv_path)


In [9]:
# ==========================================
# UNIQUE YEARS
# ==========================================

years = sorted(

    df["Year"].astype(str).unique()

)

# ==========================================
# STORE ACCURACY RESULTS
# ==========================================

accuracy_results = []

# ==========================================
# ROLLING VALIDATION
# ==========================================

for i in range(5, len(years) - 1):

    # --------------------------------------
    # TRAIN YEARS
    # --------------------------------------

    train_years = years[:i]

    # --------------------------------------
    # TEST YEAR
    # --------------------------------------

    test_year = years[i]

    print("\n===================================")
    print(f"TRAIN: {train_years[0]} → {train_years[-1]}")
    print(f"TEST YEAR: {test_year}")
    print("===================================")

    # --------------------------------------
    # TRAIN DATA
    # --------------------------------------

    train_df = df[
        df["Year"].astype(str).isin(train_years)
    ]

    # --------------------------------------
    # TEST DATA
    # --------------------------------------

    test_df = df[
        df["Year"].astype(str) == test_year
    ]

    # --------------------------------------
    # TOTAL FREQUENCY
    # --------------------------------------

    total_frequency = train_df.groupby("Topic")[
        "Frequency"
    ].sum()

    # --------------------------------------
    # RECENT YEARS
    # --------------------------------------

    recent_years = train_years[-3:]

    recent_df = train_df[
        train_df["Year"].astype(str).isin(
            recent_years
        )
    ]

    recent_frequency = recent_df.groupby("Topic")[
        "Frequency"
    ].sum()

    # --------------------------------------
    # TREND SCORES
    # --------------------------------------

    pivot_df = train_df.pivot_table(

        index="Year",

        columns="Topic",

        values="Frequency",

        aggfunc="sum"

    )

    pivot_df = pivot_df.fillna(0)

    # TREND SCORES USING REGRESSION SLOPE
    trend_scores = {}

    for topic in pivot_df.columns:
        slope, _, _, _, _ = stats.linregress(
            range(len(pivot_df[topic])),
            pivot_df[topic].values
        )
        trend_scores[topic] = slope
     # --------------------------------------
    # CREATE PREDICTION TABLE
    # --------------------------------------

    prediction_results = []

    for topic in total_frequency.index:

        freq_score = total_frequency[topic]

        trend_score = trend_scores.get(topic, 0)

        recent_score = recent_frequency.get(topic, 0)

        prediction_results.append({

            "Topic": topic,

            "Frequency": freq_score,

            "Trend": trend_score,

            "Recent": recent_score

        })

    # --------------------------------------
    # CREATE DATAFRAME
    # --------------------------------------

    prediction_df = pd.DataFrame(
        prediction_results
    )

    # --------------------------------------
    # NORMALIZATION
    # --------------------------------------

    scaler = MinMaxScaler()

    cols = [

        "Frequency",

        "Trend",

        "Recent"

    ]

    prediction_df[cols] = scaler.fit_transform(

        prediction_df[cols]

    )

    # --------------------------------------
    # PREDICTION SCORE
    # --------------------------------------

    prediction_df["Prediction Score"] = (

        prediction_df["Frequency"] * 0.6 +

        prediction_df["Trend"] * 0.1 +

        prediction_df["Recent"] * 0.3

    )

    # --------------------------------------
    # SORT PREDICTIONS
    # --------------------------------------

    prediction_df = prediction_df.sort_values(

        by="Prediction Score",

        ascending=False

    )

    # --------------------------------------
    # TOP 5 PREDICTED TOPICS
    # --------------------------------------

    TOP_K = 5

    predicted_topics = set(

        prediction_df.head(TOP_K)["Topic"]

    )

    # --------------------------------------
    # ACTUAL TOPICS
    # --------------------------------------

    actual_topic_df = test_df.groupby("Topic")[
        "Frequency"
    ].sum().reset_index()

    actual_topic_df = actual_topic_df.sort_values(

        by="Frequency",

        ascending=False

    )

    actual_topics = set(

        actual_topic_df.head(TOP_K)["Topic"]

    )

    # --------------------------------------
    # ACCURACY
    # --------------------------------------

    correct_predictions = predicted_topics.intersection(

        actual_topics

    )

    accuracy = (

        len(correct_predictions)

        / TOP_K

    ) * 100

    # --------------------------------------
    # STORE RESULTS
    # --------------------------------------

    accuracy_results.append({

        "Test Year": test_year,

        "Accuracy": accuracy

    })

    # --------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------

    print("\nPredicted Topics:")

    print(predicted_topics)

    print("\nActual Topics:")

    print(actual_topics)

    print("\nCorrect Predictions:")

    print(correct_predictions)

    print(f"\nTop-{TOP_K} Accuracy: {accuracy:.2f}%")



TRAIN: 2011 → 2015
TEST YEAR: 2016

Predicted Topics:
{'Textile Testing', 'Fiber Science', 'Man Made Fibres', 'Spinning', 'Evenness & Quality Control'}

Actual Topics:
{'Fiber Science', 'Man Made Fibres', 'Polymer Science', 'Chemical Processing', 'Evenness & Quality Control'}

Correct Predictions:
{'Fiber Science', 'Man Made Fibres', 'Evenness & Quality Control'}

Top-5 Accuracy: 60.00%

TRAIN: 2011 → 2016
TEST YEAR: 2017

Predicted Topics:
{'Fiber Science', 'Man Made Fibres', 'Polymer Science', 'Spinning', 'Evenness & Quality Control'}

Actual Topics:
{'Fiber Science', 'Man Made Fibres', 'Spinning', 'Evenness & Quality Control', 'Weaving'}

Correct Predictions:
{'Spinning', 'Fiber Science', 'Man Made Fibres', 'Evenness & Quality Control'}

Top-5 Accuracy: 80.00%

TRAIN: 2011 → 2017
TEST YEAR: 2018

Predicted Topics:
{'Fiber Science', 'Man Made Fibres', 'Polymer Science', 'Spinning', 'Evenness & Quality Control'}

Actual Topics:
{'Fiber Science', 'Polymer Science', 'Spinning', 'Evenne

In [10]:
# ==========================================
# FINAL RESULTS
# ==========================================

results_df = pd.DataFrame(
    accuracy_results
)

print("\n===================================")
print("FINAL VALIDATION RESULTS")
print("===================================")

print(results_df)



FINAL VALIDATION RESULTS
  Test Year  Accuracy
0      2016      60.0
1      2017      80.0
2      2018      80.0
3      2019      80.0
4      2020      80.0
5      2021     100.0
6      2022      60.0
7      2023      80.0
8      2024      80.0
9      2025     100.0


In [11]:
# ==========================================
# AVERAGE ACCURACY
# ==========================================

average_accuracy = results_df[
    "Accuracy"
].mean()

print("\n===================================")
print(f"AVERAGE ACCURACY: {average_accuracy:.2f}%")
print("===================================")

# ==========================================
# SAVE RESULTS
# ==========================================

output_path = r"D:\GATE_topic_predictor\validation_results.csv"

results_df.to_csv(output_path, index=False)

print("\nValidation results saved successfully!")


AVERAGE ACCURACY: 80.00%

Validation results saved successfully!


In [12]:
# ==========================================
# RANDOM BASELINE COMPARISON
# ==========================================

import random

baseline_accuracies = []

all_topics = list(df["Topic"].unique())

TOP_K = 5

# Run 1000 random trials
for _ in range(1000):

    random_predictions = set(

        random.sample(all_topics, TOP_K)

    )

    actual = set(actual_topics)

    accuracy = (

        len(random_predictions & actual)

        / TOP_K

    ) * 100

    baseline_accuracies.append(accuracy)

# Average random baseline
baseline_accuracy = (

    sum(baseline_accuracies)

    / len(baseline_accuracies)

)

print("\n===================================")
print("BASELINE COMPARISON")
print("===================================")

print(f"Random Baseline Accuracy: {baseline_accuracy:.2f}%")

print(f"Model Accuracy: {average_accuracy:.2f}%")

improvement = average_accuracy / baseline_accuracy

print(f"Improvement Over Random: {improvement:.2f}x")


BASELINE COMPARISON
Random Baseline Accuracy: 31.16%
Model Accuracy: 80.00%
Improvement Over Random: 2.57x
